In [ ]:
# Install all required libraries
!pip install openai-whisper torch transformers gradio soundfile numpy
!pip install indic-nlp-library sentencepiece protobuf

print("✅ All packages installed successfully!")

✅ All packages installed successfully!


In [ ]:
import whisper
import torch

print("🚀 Loading Whisper model for Tamil...")
# Load the model (this downloads once, then caches)
model = whisper.load_model("base")  # "tiny" is faster, "base" is more accurate

print("✅ Model loaded successfully!")

# Test with a sample (you can upload a Tamil audio file)
# First, upload a file: Click the folder icon on left → Upload → Select a Tamil audio file

# If you don't have audio, record yourself:
from google.colab import audio
print("🎤 Speak now for 5 seconds...")
recording = audio.record(5)  # Records 5 seconds
audio.export("test_tamil.wav")  # Save it

# Now transcribe
result = model.transcribe("test_tamil.wav", language="ta", task="transcribe")
print("\n📝 TAMIL TRANSCRIPTION:")
print(result["text"])

🚀 Loading Whisper model for Tamil...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 177MiB/s]


✅ Model loaded successfully!


ImportError: cannot import name 'audio' from 'google.colab' (/usr/local/lib/python3.12/dist-packages/google/colab/__init__.py)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("🔄 Loading Tamil translation models...")

# English -> Tamil model
print("Loading English to Tamil...")
en2ta_tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indictrans2-en-ta")
en2ta_model = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/indictrans2-en-ta")

# Tamil -> English model
print("Loading Tamil to English...")
ta2en_tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indictrans2-ta-en")
ta2en_model = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/indictrans2-ta-en")

print("✅ All models loaded!")

# Test translation
def translate_to_tamil(english_text):
    inputs = en2ta_tokenizer(english_text, return_tensors="pt", padding=True)
    translated = en2ta_model.generate(**inputs, max_length=128)
    tamil = en2ta_tokenizer.batch_decode(translated, skip_special_tokens=True)[0]
    return tamil

def translate_to_english(tamil_text):
    inputs = ta2en_tokenizer(tamil_text, return_tensors="pt", padding=True)
    translated = ta2en_model.generate(**inputs, max_length=128)
    english = ta2en_tokenizer.batch_decode(translated, skip_special_tokens=True)[0]
    return english

# Test
test_eng = "Hello, how are you?"
test_tam = translate_to_tamil(test_eng)
print(f"\n📝 TEST RESULTS:")
print(f"English: {test_eng}")
print(f"Tamil: {test_tam}")
print(f"Back to English: {translate_to_english(test_tam)}")

In [ ]:
import whisper
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gradio as gr
import numpy as np
import soundfile as sf
import os

# Load all models
print("🚀 Loading all models...")
whisper_model = whisper.load_model("base")

# Translation models
en2ta_tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indictrans2-en-ta")
en2ta_model = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/indictrans2-en-ta")

ta2en_tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indictrans2-ta-en")
ta2en_model = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/indictrans2-ta-en")
print("✅ All models ready!")

# Define the main function
def process_audio(audio_file, mode):
    """
    Process audio and return captions
    mode: "tamil_speech" or "english_speech"
    """
    if audio_file is None:
        return "No audio detected", "", ""

    try:
        if mode == "tamil_speech":
            # Tamil speech -> Tamil text
            result = whisper_model.transcribe(audio_file, language="ta", task="transcribe")
            tamil_text = result["text"]

            # Also translate to English for reference
            english_text = translate_to_english(tamil_text)

            return tamil_text, english_text, "Tamil captions generated"

        else:  # english_speech
            # English speech -> English text
            result = whisper_model.transcribe(audio_file, language="en", task="transcribe")
            english_text = result["text"]

            # Translate to Tamil
            tamil_text = translate_to_tamil(english_text)

            return tamil_text, english_text, "Tamil captions generated"

    except Exception as e:
        return f"Error: {str(e)}", "", ""

def translate_to_tamil(text):
    inputs = en2ta_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    translated = en2ta_model.generate(**inputs, max_length=128)
    return en2ta_tokenizer.batch_decode(translated, skip_special_tokens=True)[0]

def translate_to_english(text):
    inputs = ta2en_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    translated = ta2en_model.generate(**inputs, max_length=128)
    return ta2en_tokenizer.batch_decode(translated, skip_special_tokens=True)[0]

print("🎯 Pipeline ready! Now create the Gradio interface...")